## XGBoost Regressor Model with SHAP
This notebook uses data from `quotes_grouped.json` and tests an XGBoost model to predict the accepted price.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

### Loading the data
We load the data and filter out cases where no price was accepted.

In [ ]:
# Loading grouped data by quote
df = pd.read_json("../data/quotes_grouped.json")

# We only keep lines with accepted prices
df = df[df["proposed_price_accepted"].notnull()].copy()

# Remove outliers with suspiciously low prices (e.g. < 50 €)
df = df[df["proposed_price_accepted"] >= 50]

# We extract the 1st language pair and 1st document type per quote
df["source_language"] = df["languages"].apply(lambda x: x[0][0] if isinstance(x, list) and len(x) > 0 else None)
df["target_language"] = df["languages"].apply(lambda x: x[0][1] if isinstance(x, list) and len(x) > 0 else None)
df["document_type"] = df["document_types"].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)

# Converting categories to codes for XGBoost + SHAP
for col in ["source_language", "target_language", "unit", "document_type"]:
    df[col] = df[col].astype("category").cat.codes

# New features

df["num_languages"] = df["languages"].apply(len)
df["has_rare_doc"] = df["document_types"].apply(lambda lst: any("contract" not in d.lower() for d in lst))

# Define features
X = df[[
    "source_language", "target_language", "unit",
    "number_of_unit", "document_type", "requested_deadline",
    "num_languages", "has_rare_doc"
]]

# Logarithmic transformation of the target to reduce the effect of extreme values
y = np.log1p(df["proposed_price_accepted"])  # log(1 + x)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
y_test = y_test.reset_index(drop=True)


### Training the XGBoost Regressor model with different hyperparameters (GridSearchCV)

In [ ]:
# Parameters to test
param_grid = {
    'n_estimators': [300, 400, 500],
    'max_depth': [6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1]
}

# Basic model
xgb_model = xgb.XGBRegressor(objective="reg:squarederror", random_state=42)

# GridSearch with cross-validation (3 folds)
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=3,
    scoring='neg_mean_absolute_error',
    verbose=1,
    n_jobs=-1
)

# Execution
grid_search.fit(X_train, y_train)

# Best settings and score
print("Best hyperparameters :", grid_search.best_params_)
print(f"Average MAE on folds : {-grid_search.best_score_:.2f} €")

# Retrain the best model on the entire training set
best_model = grid_search.best_estimator_
best_model.fit(X_train, y_train)

### Model Evaluation

In [ ]:
# Inversion of the logarithmic transformation
y_test_exp = np.expm1(y_test)
y_pred_exp = np.expm1(best_model.predict(X_test))

# MAPE : average error percentage
mape = np.mean(np.abs((y_test_exp - y_pred_exp) / y_test_exp)) * 100
print(f"MAPE : {mape:.2f} %")

# Weighted MAE: more weight given to large amounts
mae_weighted = np.average(np.abs(y_test - best_model.predict(X_test)), weights=y_test)
print(f"Weighted MAE (log-scale) : {mae_weighted:.2f}")

In [ ]:
for real, pred in zip(y_test_exp[:10], y_pred_exp[:10]):
    print(f"Real: {real:.2f} € | Predicted: {pred:.2f} € | Diff: {abs(real - pred):.2f} €")

### Interpretation with SHAP

In [ ]:
# SHAP interpretation (log scale)
X_train_enc = X_train.copy().astype(float)

explainer = shap.Explainer(best_model, X_train_enc)
shap_values = explainer(X_train_enc)

shap.plots.bar(shap_values, max_display=10)

In [ ]:
comparison_df = pd.DataFrame({
    "Actual price (€)": np.expm1(y_test).values,
    "Predicted price (€)": np.expm1(best_model.predict(X_test)),
})

comparison_df["Absolute error (€)"] = (comparison_df["Actual price (€)"] - comparison_df["Predicted price (€)"]).abs()
comparison_df["Within 5€?"] = comparison_df["Absolute error (€)"] <= 5

print(comparison_df.head(30))

In [ ]:
# Comparison between actual and predicted prices

comparison_df = pd.DataFrame({
    "Actual price (€)": np.expm1(y_test).values,
    "Predicted price (€)": np.expm1(best_model.predict(X_test))
})

# Displaying the first 30 lines
comparison_df.head(30)

In [ ]:
df["proposed_price_accepted"].hist(bins=50)
plt.title("Distribution of accepted prizes")
plt.xlabel("Price (€)")
plt.ylabel("Number of quotes")
plt.show()